In [2]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

In [3]:
data = fetch_california_housing()
X = data.data
y = data.target

In [4]:
feature_names = data.feature_names
print("Feature names:", feature_names)

Feature names: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']


In [5]:
train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
class LinearRegression1D:
    def __init__(self):
        self.w1 = 1
        self.w0 = 0
    
    def predict(self, X):
        return X * self.w1 + self.w0

    def fit(self, X, y):
        self.w1 = self._learn_w1(X, y)
        self.w0 = self._learn_w0(X, y, self.w1)

    def _learn_w1(self, X, y):
        x_bar = np.mean(X)
        y_bar = np.mean(y)
        return np.sum((X - x_bar) * (y - y_bar)) / np.sum((X - x_bar) ** 2)
    
    def _learn_w0(self, X, y, w_1):
        x_bar = np.mean(X)
        y_bar = np.mean(y)
        return y_bar - self.w1 * x_bar


In [7]:
model = LinearRegression1D()

In [8]:
train_X_0 = train_X[:, 0]
model.fit(train_X_0, train_y)

In [9]:
test_X_0 = test_X[:, 0]
predictions = model.predict(test_X_0)

In [10]:
score = np.mean((predictions - test_y) ** 2)
print(f"Mean Squared Error: {score}")

Mean Squared Error: 0.7091157771765549


In [11]:
for feat in range(train_X.shape[1]):
    model = LinearRegression1D()
    train_X_feat = train_X[:, feat]
    model.fit(train_X_feat, train_y)
    test_X_feat = test_X[:, feat]
    predictions = model.predict(test_X_feat)
    score = np.mean((predictions - test_y) ** 2)
    name = feature_names[feat]
    print(f"Feature {name} (Index {feat}): Mean Squared Error: {score}")

Feature MedInc (Index 0): Mean Squared Error: 0.7091157771765549
Feature HouseAge (Index 1): Mean Squared Error: 1.2939617265100323
Feature AveRooms (Index 2): Mean Squared Error: 1.2923314440807299
Feature AveBedrms (Index 3): Mean Squared Error: 1.3108875538359483
Feature Population (Index 4): Mean Squared Error: 1.3102870667503983
Feature AveOccup (Index 5): Mean Squared Error: 1.3096449354773076
Feature Latitude (Index 6): Mean Squared Error: 1.281690431624196
Feature Longitude (Index 7): Mean Squared Error: 1.3081058483312469


In [12]:
class LinearRegressionND:
    def __init__(self):
        self.theta = None
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        bias = np.ones((n_samples, 1))
        X = np.hstack((bias, X))
        self.theta = np.linalg.inv(X.T @ X) @ X.T @ y

    def predict(self, X):
        bias = np.ones((X.shape[0], 1))
        X = np.hstack((bias, X))
        return X @ self.theta

In [13]:
model = LinearRegressionND()
model.fit(train_X, train_y)
predictions = model.predict(test_X)
score = np.mean((predictions - test_y) ** 2)
print(f"Mean Squared Error: {score}")

Mean Squared Error: 0.5558915986954801


In [14]:
dp = 4  # set desired decimal places

param_names = ["Bias"] + feature_names
rounded_theta = np.round(model.theta, dp)

for name, value in zip(param_names, rounded_theta):
    print(f"{name}: {value:.{dp}f}")

Bias: -37.0233
MedInc: 0.4487
HouseAge: 0.0097
AveRooms: -0.1233
AveBedrms: 0.7831
Population: -0.0000
AveOccup: -0.0035
Latitude: -0.4198
Longitude: -0.4337


In [18]:
class PolynomialRegression:
    def __init__(self, degree):
        self.degree = degree
        self.theta = None
    
    def fit(self, X, y):
        design_matrix = self._design_matrix(X)
        self.theta = np.linalg.inv(design_matrix.T @ design_matrix) @ design_matrix.T @ y
    
    def _design_matrix(self, X):
        "returns the design matrix for the given input matrix X, it includes the linear terms"
        n_samples, _ = X.shape
        bias = np.ones((n_samples, 1))
        X_poly = bias
        for d in range(1, self.degree + 1):
            X_poly = np.hstack((X_poly, X ** d))
        return X_poly

    def predict(self, X):
        design_matrix = self._design_matrix(X)
        return design_matrix @ self.theta

In [19]:
model = PolynomialRegression(degree=2)
model.fit(train_X, train_y)
predictions = model.predict(test_X)
score = np.mean((predictions - test_y) ** 2)
print(f"Mean Squared Error: {score}")

Mean Squared Error: 0.8420482078600596


In [20]:
dp = 4  # set desired decimal places

param_names = ["Bias"] + feature_names
rounded_theta = np.round(model.theta, dp)

for name, value in zip(param_names, rounded_theta):
    print(f"{name}: {value:.{dp}f}")

Bias: -245.6576
MedInc: 0.6322
HouseAge: -0.0039
AveRooms: -0.2168
AveBedrms: 1.4466
Population: -0.0000
AveOccup: -0.0121
Latitude: -1.8421
Longitude: -4.3322


In [21]:
class PolynomialRegressionWithInteractions:
    def __init__(self, degree):
        self.degree = degree
        self.theta = None
    
    def fit(self, X, y):
        design_matrix = self._design_matrix(X)
        self.theta = np.linalg.inv(design_matrix.T @ design_matrix) @ design_matrix.T @ y
    
    def _design_matrix(self, X):
        "returns the design matrix for the given input matrix X, it includes the linear terms and interaction terms"
        n_samples, n_features = X.shape
        bias = np.ones((n_samples, 1))
        X_poly = bias
        for d in range(1, self.degree + 1):
            X_poly = np.hstack((X_poly, X ** d))
        
        # Add interaction terms
        for i in range(n_features):
            for j in range(i + 1, n_features):
                interaction_term = (X[:, i] * X[:, j]).reshape(-1, 1)
                X_poly = np.hstack((X_poly, interaction_term))
        
        return X_poly

    def predict(self, X):
        design_matrix = self._design_matrix(X)
        return design_matrix @ self.theta

In [22]:
model = PolynomialRegressionWithInteractions(degree=2)
model.fit(train_X, train_y)
predictions = model.predict(test_X)
score = np.mean((predictions - test_y) ** 2)
print(f"Mean Squared Error: {score}")

Mean Squared Error: 0.464301572861215


In [29]:
for degree in range(1, 10):
    model = PolynomialRegression(degree=degree)
    model.fit(train_X, train_y)
    predictions_test = model.predict(test_X)
    score_test = np.mean((predictions_test - test_y) ** 2)
    predictions_train = model.predict(train_X)
    score_train = np.mean((predictions_train - train_y) ** 2)
    print(f"Degree {degree}: Mean squared Error on Training: {score_train} Mean Squared Error on Test: {score_test}")

Degree 1: Mean squared Error on Training: 0.5179331255246699 Mean Squared Error on Test: 0.5558915986954801
Degree 2: Mean squared Error on Training: 0.4947499844707385 Mean Squared Error on Test: 0.8420482078600596
Degree 3: Mean squared Error on Training: 0.45312021808663405 Mean Squared Error on Test: 7.057760336648495
Degree 4: Mean squared Error on Training: 0.4292359523333788 Mean Squared Error on Test: 289.0673126238734
Degree 5: Mean squared Error on Training: 0.5621469426867821 Mean Squared Error on Test: 8066.317362278326
Degree 6: Mean squared Error on Training: 15.177536891379177 Mean Squared Error on Test: 98724.1239861205
Degree 7: Mean squared Error on Training: 32.359827832087646 Mean Squared Error on Test: 6657796.466083872
Degree 8: Mean squared Error on Training: 116.2074380590001 Mean Squared Error on Test: 2377103.378999781
Degree 9: Mean squared Error on Training: 636.4690295419297 Mean Squared Error on Test: 276356079750.6582


In [30]:
for degree in range(1, 10):
    model = PolynomialRegressionWithInteractions(degree=degree)
    model.fit(train_X, train_y)
    predictions_test = model.predict(test_X)
    score_test = np.mean((predictions_test - test_y) ** 2)
    predictions_train = model.predict(train_X)
    score_train = np.mean((predictions_train - train_y) ** 2)
    print(f"Degree {degree}: Mean squared Error on Training: {score_train} Mean Squared Error on Test: {score_test}")

Degree 1: Mean squared Error on Training: 0.44508895586534236 Mean Squared Error on Test: 0.4946973440923558
Degree 2: Mean squared Error on Training: 0.42072661515742565 Mean Squared Error on Test: 0.464301572861215
Degree 3: Mean squared Error on Training: 0.40628703586092474 Mean Squared Error on Test: 2.0574589788450317
Degree 4: Mean squared Error on Training: 0.41156575869485496 Mean Squared Error on Test: 176.59197967225722
Degree 5: Mean squared Error on Training: 0.4090883050155595 Mean Squared Error on Test: 5251.5604349911
Degree 6: Mean squared Error on Training: 22.606522362229274 Mean Squared Error on Test: 393724.4657367489
Degree 7: Mean squared Error on Training: 108.58376585523382 Mean Squared Error on Test: 14796417.130077677
Degree 8: Mean squared Error on Training: 581.5474765069955 Mean Squared Error on Test: 33033101.357244547
Degree 9: Mean squared Error on Training: 154.42741375919786 Mean Squared Error on Test: 1736271618985.349


In [32]:
from sklearn.datasets import fetch_openml

In [36]:
data = fetch_openml(name="mauna-loa-atmospheric-co2", as_frame=False, parser='liac-arff')
print(data)

{'data': array([[1958.,    3.,   29.,    4.,    0.,    0.],
       [1958.,    4.,    5.,    6.,    0.,    0.],
       [1958.,    4.,   12.,    4.,    0.,    0.],
       ...,
       [2001.,   12.,   15.,    7.,    0.,    0.],
       [2001.,   12.,   22.,    6.,    0.,    0.],
       [2001.,   12.,   29.,    6.,    0.,    0.]], shape=(2225, 6)), 'target': array([316.1, 317.3, 317.6, ..., 371.2, 371.3, 371.5], shape=(2225,)), 'frame': None, 'categories': {'station': ['MLO']}, 'feature_names': ['year', 'month', 'day', 'weight', 'flag', 'station'], 'target_names': ['co2'], 'DESCR': '**Weekly carbon-dioxide concentration averages derived from continuous air samples for the Mauna Loa Observatory, Hawaii, U.S.A.**<br><br>\nThese weekly averages are ultimately based on measurements of 4 air samples per hour taken atop intake lines on several towers during steady periods of CO2 concentration of not less than 6 hours per day; if no such periods are available on a given day, then no data are used 